In [1]:
import psycopg2
import pandas as pd

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    try:
        connection = psycopg2.connect(
            host=host_ip, database=database_name, user=user, password=password, port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")
        df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")
        return df
    except Exception as e:
        print(f"❌ Error: {e}")
        return None
    finally:
        if 'connection' in locals():
            connection.close()

# --- Configuration ---
HOST_IP = "100.94.14.115"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

Connected successfully to pradigma-extractor on 100.94.14.115


C:\Users\win 11\AppData\Local\Temp\ipykernel_5664\1965381518.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(f"SELECT * FROM {table_name};", connection)


✅ Fetched 7990 rows from 'extraction'


In [37]:
df = df_original.copy(deep=True)
df = df[
    (df['dept_name'] == 'Track-Network') & (df['status_id'] == 1)
][['filename', 'workorder_id', 'interval', 'dept_name', 'json_data']]

df

,filename,workorder_id,interval,dept_name,json_data
427,TN_PM_MTH_BogieTestJig_4000512660.pdf,4.000513e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
428,TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,4.000496e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
473,TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,4.000485e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
640,TN_PM_MTH_MechanicalSwitch1_4000490559.pdf,4.000491e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
803,TN_PM_MTH_PowerRail_4000541408.pdf,4.000541e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...,...,...
7583,TN_PM_MTH_Walkway_4000492095.pdf,4.000492e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
7599,TN_PM_MTH_FingerPlateDownBeam_4000589669.pdf,4.000590e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
7600,TN_PM_MTH_FingerPlateUpBeam_4000589645.pdf,4.000590e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."
7601,TN_PM_MTH_FingerPlateUpBeam_4000589646.pdf,4.000590e+09,Monthly,Track-Network,"{'notification': {'notification_no': 'NA', 'no..."


In [38]:
import json

def normalize(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return None
    return None

df["json_data"] = df["json_data"].apply(normalize)

def safe_get(d, keys):
    for k in keys:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
    return d

def find_first_key(data, keys):
    if isinstance(data, dict):
        for k in keys:
            if k in data:
                return data[k]

        for v in data.values():
            result = find_first_key(v, keys)
            if result is not None:
                return result

    elif isinstance(data, list):
        for item in data:
            result = find_first_key(item, keys)
            if result is not None:
                return result

    return None

extracted_df = pd.DataFrame({

    "workorder_no": df["workorder_id"],

    "inspection_date": df["json_data"].apply(
        lambda x: find_first_key(x, ["date_closed"])
    ),

    "verified_by": df["json_data"].apply(
        lambda x: find_first_key(x, ["supervisor", "supervisor_id"])
    ),

    "performed_by": df["json_data"].apply(
        lambda x: find_first_key(x, ["technician", "technicians", "signature"])
    ),

    "filename": df["filename"],
})

extracted_df.head()

,workorder_no,inspection_date,verified_by,performed_by,filename
427,4.000513e+09,15/2/23,7091,"{'signature': '11625, 9728', 'date': '15/02/20...",TN_PM_MTH_BogieTestJig_4000512660.pdf
428,4.000496e+09,13/11/2022,7091,"{'technicians': '9728, 11625', 'date': '13/11/...",TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf
473,4.000485e+09,NA,"{'supervisor_id': 9264, 'date': '17/09/2022'}","{'technician_id': '6453, 11626', 'date': '17/0...",TN_PM_MTH_MechanicalSwitch1_4000484591.pdf
640,4.000491e+09,NA,"{'supervisor_id': '7091', 'date': '17/10/2022'}","{'technician_id': '9726, 11625', 'date': '17/1...",TN_PM_MTH_MechanicalSwitch1_4000490559.pdf
803,4.000541e+09,NA,"{'supervisor_id': '7019', 'date': '31/07/2023'}","{'technician_id': '11590, 9728, 20003, 10180',...",TN_PM_MTH_PowerRail_4000541408.pdf


In [39]:
def extract_tech_ids(x):
    if not isinstance(x, dict):
        return None

    value = x.get("signature") or x.get("technicians") or x.get("technician_id") or x.get("technicians_id") or x.get("id")

    if not isinstance(value, str):
        return None

    # clean spaces
    return ",".join([i.strip() for i in value.split(",") if i.strip()])

def extract_supervisor_ids(x):

    if isinstance(x, (int, str)):
        return str(x).strip()

    if isinstance(x, dict):
        value = x.get("supervisor_id") or x.get("id")
        return str(value).strip() if value is not None else None

    return None

extracted_df["technician_ids"] = extracted_df["performed_by"].apply(extract_tech_ids)
extracted_df["supervisor_id"] = extracted_df["verified_by"].apply(extract_supervisor_ids)

extracted_df.head()

,workorder_no,inspection_date,verified_by,performed_by,filename,technician_ids,supervisor_id
427,4.000513e+09,15/2/23,7091,"{'signature': '11625, 9728', 'date': '15/02/20...",TN_PM_MTH_BogieTestJig_4000512660.pdf,"11625,9728",7091
428,4.000496e+09,13/11/2022,7091,"{'technicians': '9728, 11625', 'date': '13/11/...",TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,"9728,11625",7091
473,4.000485e+09,NA,"{'supervisor_id': 9264, 'date': '17/09/2022'}","{'technician_id': '6453, 11626', 'date': '17/0...",TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,"6453,11626",9264
640,4.000491e+09,NA,"{'supervisor_id': '7091', 'date': '17/10/2022'}","{'technician_id': '9726, 11625', 'date': '17/1...",TN_PM_MTH_MechanicalSwitch1_4000490559.pdf,"9726,11625",7091
803,4.000541e+09,NA,"{'supervisor_id': '7019', 'date': '31/07/2023'}","{'technician_id': '11590, 9728, 20003, 10180',...",TN_PM_MTH_PowerRail_4000541408.pdf,"11590,9728,20003,10180",7019


In [40]:
extracted_df.to_excel("extracted/extracted_tnm.xlsx", index=False)

### Comparing list of technician and supervisor with tbl_user

In [41]:
ref_user = pd.read_excel("tbl_users.xlsx")
ref_user.head()

,id,staff_id,name,call_sign,position,department,email
0,1,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Senior Associate,Power System,luqnman.jamaludin@prasarana.com.my
1,2,10007223,ROHAIZAN BIN MASTOR,RM7223,Associate,Power System,rohaizan@prasarana.com.my
2,3,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Associate,Power System,nurzamzam.rosle@prasarana.com.my
3,4,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Associate,Power System,idhar.azhar@prasarana.com.my
4,5,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Senior Associate,Power System,ikhwan.abdullah@prasarana.com.my


In [42]:
cleaned_user = ref_user[["staff_id", "name", "call_sign", "department"]].copy()

cleaned_user["stamp_id"] = (
    cleaned_user["staff_id"]
    .astype(str)
    .str.replace(r"^1000|^100", "", regex=True)
)

cleaned_user["stamp_id"] = cleaned_user["stamp_id"].astype(str)
cleaned_user.head()

,staff_id,name,call_sign,department,stamp_id
0,10018972,LUQMAN NULHAKIM BIN JAMALUDIN,LNJ 18972,Power System,18972
1,10007223,ROHAIZAN BIN MASTOR,RM7223,Power System,7223
2,10023969,MUHAMMAD NUR ZAM ZAM BIN MOHD ROSLE,NZZ23969,Power System,23969
3,10025298,IDHAR DANIEL BIN MOHD AZHAR,IDA25298,Power System,25298
4,10024324,MUHAMMAD IKHWAN BIN ABDULLAH,MIA24324,Power System,24324


In [43]:
import pandas as pd

df_tnm = pd.read_excel("extracted/extracted_tnm.xlsx")
df_tnm.head()

,workorder_no,inspection_date,verified_by,performed_by,filename,technician_ids,supervisor_id
0,4.000513e+09,15/2/23,7091,"{'signature': '11625, 9728', 'date': '15/02/20...",TN_PM_MTH_BogieTestJig_4000512660.pdf,"11625,9728",7091
1,4.000496e+09,13/11/2022,7091,"{'technicians': '9728, 11625', 'date': '13/11/...",TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,"9728,11625",7091
2,4.000485e+09,NaN,"{'supervisor_id': 9264, 'date': '17/09/2022'}","{'technician_id': '6453, 11626', 'date': '17/0...",TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,"6453,11626",9264
3,4.000491e+09,NaN,"{'supervisor_id': '7091', 'date': '17/10/2022'}","{'technician_id': '9726, 11625', 'date': '17/1...",TN_PM_MTH_MechanicalSwitch1_4000490559.pdf,"9726,11625",7091
4,4.000541e+09,NaN,"{'supervisor_id': '7019', 'date': '31/07/2023'}","{'technician_id': '11590, 9728, 20003, 10180',...",TN_PM_MTH_PowerRail_4000541408.pdf,"11590,9728,20003,10180",7019


In [44]:
id_to_name = dict(zip(
    cleaned_user["stamp_id"].astype(str),
    cleaned_user["name"]
))

id_to_supervisor_name = id_to_name

def build_rows(row):
    
    tech_ids = row["technician_ids"]

    if not isinstance(tech_ids, str):
        return []

    tech_list = [i.strip() for i in tech_ids.split(",") if i.strip()]

    results = []

    for tech_id in tech_list:
        
        results.append({
            "filename": row.get("filename"),
            "workorder_no": row.get("workorder_no"),
            "inspection_date": row.get("inspection_date"),

            "technician_id": tech_id,
            "name": id_to_name.get(tech_id, ""),

            "supervisor_id": row.get("supervisor_id"),
            "supervisor_name": id_to_supervisor_name.get(
                str(row.get("supervisor_id")), ""
            ),
        })

    return results

expanded = df_tnm.apply(build_rows, axis=1).explode().dropna()

final_df = pd.DataFrame(expanded.tolist())
final_df.head()

,filename,workorder_no,inspection_date,technician_id,name,supervisor_id,supervisor_name
0,TN_PM_MTH_BogieTestJig_4000512660.pdf,4.000513e+09,15/2/23,11625,MUHAMMAD ZARUL FIRDAUS BIN ZULKIFLI,7091,MAHADI BIN MOHAMAD
1,TN_PM_MTH_BogieTestJig_4000512660.pdf,4.000513e+09,15/2/23,9728,MOHAMMAD HAFIFI BIN RIMI,7091,MAHADI BIN MOHAMAD
2,TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,4.000496e+09,13/11/2022,9728,MOHAMMAD HAFIFI BIN RIMI,7091,MAHADI BIN MOHAMAD
3,TN_PM_MTH_BogieDropPit1Lub_4000495745.pdf,4.000496e+09,13/11/2022,11625,MUHAMMAD ZARUL FIRDAUS BIN ZULKIFLI,7091,MAHADI BIN MOHAMAD
4,TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,4.000485e+09,NaN,6453,MUHAMMAD NAWAWI BIN ABDUL RAHMAN,9264,


In [45]:
final_df.to_excel("output/staff_tnm.xlsx", index=False)